In [27]:
jogo = 18244

In [ ]:
import json
from conexao import conectar

eventPath = f"../../data/raw/events/{jogo}.json"
with open(eventPath, encoding="utf-8") as f:
    events = json.load(f)

sql = """
    INSERT INTO events (
        id, game, idx, period, minute, second, action, player_id, position,
        location_x, location_y,
        pass_length, pass_outcome,
        shot_xg, shot_outcome, shot_technique,
        duel_type, duel_outcome,
        key_pass_id
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = []
for event in events:
    if not event.get("player"):
        continue

    location  = event.get("location") or [None, None]
    pass_data = event.get("pass", {})
    shot_data = event.get("shot", {})
    duel_data = event.get("duel", {})

    rows.append((
        event["id"],
        jogo,
        event.get("index"),
        event.get("period"),
        event.get("minute"),
        event.get("second"),
        event.get("type", {}).get("name"),
        event["player"]["id"],
        event.get("position", {}).get("name"),
        location[0],
        location[1],
        pass_data.get("length"),
        pass_data.get("outcome", {}).get("name"),
        shot_data.get("statsbomb_xg"),
        shot_data.get("outcome", {}).get("name"),
        shot_data.get("technique", {}).get("name"),
        duel_data.get("type", {}).get("name"),
        duel_data.get("outcome", {}).get("name"),
        shot_data.get("key_pass_id"),
    ))

conn = conectar()
try:
    cursor = conn.cursor()
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"{cursor.rowcount} eventos inseridos.")
finally:
    cursor.close()
    conn.close()


3388 eventos inseridos.


: 